# Best Practices — Pipeline

This is the main job notebook. It is responsible for the I/O and the orchestration:

1. create the `SparkSession`
2. pull in the transformation library via `%run "./Best Practices - Functions.ipynb"`
3. read the inputs (with the `score > 0` filter applied at read time)
4. apply the transformations and run the actions
5. shut the session down

In a real codebase this would be a `.py` entry point script (`jobs/daily_user_stats.py` or similar) that imports `compute_user_region_stats` from a regular Python module. Here we use `%run` to include the Functions notebook in the same kernel.

In [ ]:
from pyspark.sql import SparkSession

import os

In [ ]:
spark = (
    SparkSession
    .builder
    .appName('Best Practices - Pipeline')
    .getOrCreate()
)

#### Include the Functions notebook

`%run` executes the included notebook in the current kernel, so after this cell `f`, `DataFrame`, `location_to_region` and `compute_user_region_stats` are all available.

In [ ]:
%run "./Best Practices - Functions.ipynb"

In [ ]:
base_path = os.getcwd()

project_path = ('/').join(base_path.split('/')[0:-3]) 

users_input_path = os.path.join(project_path, 'data/users')
answers_input_path = os.path.join(project_path, 'data/answers')

#### Read the inputs

The `score > 0` filter is applied directly on the read instead of after a cache — there is no point in caching rows that are about to be filtered out.

In [ ]:
usersDF = spark.read.parquet(users_input_path)

answersDF = spark.read.parquet(answers_input_path).filter(f.col('score') > 0)

#### Compute and run the actions

Three actions consume `result` (`orderBy(...).show()`, `groupBy(...).count().show()`, and the `noop` write). Without caching the entire upstream pipeline would run three times, so `result` gets a `.cache()` here. `.unpersist()` at the end releases the storage.

In [ ]:
result = compute_user_region_stats(usersDF, answersDF, min_answers=5).cache()

result.orderBy(f.desc('avg_score')).show(10)
result.groupBy('region').count().show()
result.write.mode('overwrite').format('noop').save()

result.unpersist()

In [ ]:
spark.stop()